# LoRA Early Stopping Lesson

This notebook walks through a small reinforcement fine-tuning example using GRPO and LoRA on a simple arithmetic task, then adds validation-based early stopping so training can halt once held-out reward stops improving.


## Imports

Import the standard library, PyTorch, dataset tools, Transformers components, and the TRL and PEFT classes used throughout the lesson, including a custom trainer callback for moving-average early stopping.


In [1]:
import re
from collections import deque
from typing import Any
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import shutil

for directory in ["grpo-arithmetic-lora-early-stopping-demo", "grpo-arithmetic-lora-early-stopping-adapter"]:
    shutil.rmtree(Path(directory), ignore_errors=True)

print("Deleted any existing GRPO early-stopping output directories.")


Deleted any existing GRPO early-stopping output directories.


## Constants

Define the dataset bounds and the pretrained instruction model that will be evaluated and then fine-tuned.

In [3]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"


## Dataset Builder

Create a helper function that generates arithmetic prompts and the expected answers for a small synthetic training set.

In [4]:
def make_dataset() -> Dataset:
    """Build a small arithmetic dataset with strict output-format instructions.

    Args:
        None.

    Returns:
        Dataset: A Hugging Face dataset containing prompt and answer pairs.
    """
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            rows.append({
                "prompt": f"What is {a} + {b}? Respond exactly as <think>...</think><answer>...</answer>",
                "answer": str(a + b),
            })

    return Dataset.from_list(rows)


## Dataset Split

Build the dataset and split it into train, validation, and test subsets so we can use validation reward for early stopping while keeping the test set reserved for final before-and-after evaluation.


In [5]:
dataset = make_dataset()
test_split = dataset.train_test_split(test_size=0.25, seed=42)
train_validation_dataset = test_split["train"]
test_dataset = test_split["test"]

train_validation_split = train_validation_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_validation_split["train"]
validation_dataset = train_validation_split["test"]

# Keep the first 5 training prompts so we can filter GRPO completion logs later.
num_first_train_records = min(5, len(train_dataset))
first_five_train_prompts = set(train_dataset.select(range(num_first_train_records))["prompt"])

print(f"Dataset sizes -> train: {len(train_dataset)}, validation: {len(validation_dataset)}, test: {len(test_dataset)}")

def show_examples(name: str, split_dataset: Dataset, limit: int = 3) -> None:
    num_examples = min(limit, len(split_dataset))
    print(f"\nFirst {num_examples} examples from the {name} dataset:")
    for i in range(num_examples):
        print(split_dataset[i])

show_examples("train", train_dataset)
show_examples("validation", validation_dataset)
show_examples("test", test_dataset)


Dataset sizes -> train: 120, validation: 30, test: 50

First 3 examples from the train dataset:
{'prompt': 'What is 19 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '22'}
{'prompt': 'What is 20 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '23'}
{'prompt': 'What is 8 + 7? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '15'}

First 3 examples from the validation dataset:
{'prompt': 'What is 10 + 4? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '14'}
{'prompt': 'What is 17 + 2? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 12 + 1? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '13'}

First 3 examples from the test dataset:
{'prompt': 'What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>', 'a

## Answer Extraction Helper

Define a parser that pulls the contents of the `<answer>` tag from a generated response.

In [6]:
def extract_answer(text: str) -> str:
    """Return the contents of the first <answer> tag, or an empty string.

    Args:
        text: The generated model response to parse.

    Returns:
        str: The extracted answer text, or an empty string if no answer tag exists.
    """
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else ""


## Format Validation Helper

Check whether a model response follows the required output structure with both `<think>` and `<answer>` tags.

In [7]:
def has_required_format(text: str) -> bool:
    """Check whether the response contains both think and answer tags.

    Args:
        text: The generated model response to validate.

    Returns:
        bool: True when the response includes both required tags, otherwise False.
    """
    return bool(re.search(
        r"<think>.*?</think>\s*<answer>.*?</answer>",
        text,
        re.DOTALL
    ))


## Format Reward Function

Assign a reward to each completion based on whether it follows the required tagged response format.

In [8]:
def format_reward(completions: list, **kwargs: Any) -> list[float]:
    """Reward completions that follow the required XML-like response format.

    Args:
        completions: Generated responses returned by the trainer or model.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One format reward per completion.
    """
    rewards = []

    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(0.5 if has_required_format(text) else 0.0)

    return rewards


## Correctness Reward Function

Assign a reward to each completion based on whether the extracted answer matches the expected target value.

In [9]:
def correctness_reward(completions: list, answer: list[str], **kwargs: Any) -> list[float]:
    """Reward completions whose extracted answer matches the expected answer.

    Args:
        completions: Generated responses returned by the trainer or model.
        answer: Expected answer strings aligned with the completions.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One correctness reward per completion.
    """
    rewards = []

    for c, expected in zip(completions, answer):
        text = c[0]["content"] if isinstance(c, list) else c
        predicted = extract_answer(text)
        rewards.append(1.0 if predicted == expected else 0.0)

    return rewards


## Response Generation Helper

Define a helper that formats a prompt as a chat conversation, runs generation, and decodes only the new tokens.

In [10]:
def generate_response(model: Any, tokenizer: Any, prompt: str) -> str:
    """Generate a deterministic response for a single user prompt.

    Args:
        model: The causal language model used for generation.
        tokenizer: The tokenizer used to format and decode the prompt.
        prompt: The user prompt to send to the model.

    Returns:
        str: The decoded generated response text.
    """
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


## Evaluation Helper

Define the evaluation routine that generates predictions across the test set and prints accuracy, format compliance, and sample outputs.

In [11]:
def evaluate_model(model: Any, tokenizer: Any, eval_dataset: Dataset, label: str) -> None:
    """Run evaluation on a dataset and print summary metrics with examples.

    Args:
        model: The causal language model to evaluate.
        tokenizer: The tokenizer paired with the model.
        eval_dataset: The evaluation split containing prompts and answers.
        label: A display label for the evaluation output.

    Returns:
        None: This function prints metrics and sample generations.
    """
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    total_reward = 0.0

    examples = []

    for row in eval_dataset:
        prompt = row["prompt"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        predicted = extract_answer(text)

        is_formatted = has_required_format(text)
        is_correct = predicted == expected

        format_score = 0.5 if is_formatted else 0.0
        correctness_score = 1.0 if is_correct else 0.0
        reward = format_score + correctness_score

        formatted += int(is_formatted)
        correct += int(is_correct)
        total_reward += reward

        if len(examples) < 5:
            examples.append({
                "prompt": prompt,
                "expected": expected,
                "generated": text,
                "predicted": predicted,
                "reward": reward,
            })

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {correct / total:.2%}")
    print(f"Format compliance: {formatted}/{total} = {formatted / total:.2%}")
    print(f"Average reward:    {total_reward / total:.3f}")

    print("\nSample generations:")
    for ex in examples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Reward:   ", ex["reward"])


## Tokenizer Setup

Load the tokenizer for the base instruction model so prompts can be formatted and outputs decoded.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Base Model Setup

Load the pretrained causal language model and choose a practical dtype depending on whether CUDA is available.

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1459.39it/s]


## Baseline Evaluation

Measure how the base model performs on the held-out arithmetic examples before applying GRPO and LoRA.

In [14]:
evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before GRPO + LoRA"
)



=== Before GRPO + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Average reward:    0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3 = 19</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7 = 10</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6 = 18</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  12
Generated: <think>5 + 7 = 12</

## LoRA Configuration

Configure the LoRA adapter modules and hyperparameters that will be attached during GRPO training.

In [15]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## Early Stopping Overview

Early stopping watches a validation metric during training and halts the run once that metric stops improving for a chosen patience window. In this notebook, GRPO evaluates on the validation split every few steps, computes a moving average of `eval_reward`, and uses that smoothed metric to decide when progress has stalled. The test set is not used for stopping so it remains a clean final measurement.


## GRPO Training Arguments

Set the GRPO hyperparameters, including output location, batch sizes, number of generations, completion length, and the evaluation and checkpoint settings needed for moving-average early stopping.


In [16]:
moving_average_window = 3
early_stopping_patience = 2
early_stopping_threshold = 0.0

class MovingAverageEarlyStoppingCallback(TrainerCallback):
    """Stop training when the moving average of eval_reward stops improving."""

    def __init__(self, window_size: int, patience: int, threshold: float = 0.0):
        self.window_size = window_size
        self.patience = patience
        self.threshold = threshold
        self.history: deque[float] = deque(maxlen=window_size)
        self.best_smoothed = None
        self.best_step = None
        self.bad_eval_count = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.history.clear()
        self.best_smoothed = None
        self.best_step = None
        self.bad_eval_count = 0

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        reward = metrics.get("eval_reward")
        if reward is None:
            return control

        self.history.append(float(reward))
        smoothed_reward = sum(self.history) / len(self.history)
        metrics["eval_smoothed_reward"] = smoothed_reward
        metrics["eval_smoothed_reward_window"] = len(self.history)

        if self.best_smoothed is None or smoothed_reward > self.best_smoothed + self.threshold:
            self.best_smoothed = smoothed_reward
            self.best_step = state.global_step
            self.bad_eval_count = 0
        else:
            self.bad_eval_count += 1

        metrics["eval_smoothed_reward_best"] = self.best_smoothed
        metrics["eval_smoothed_reward_bad_eval_count"] = self.bad_eval_count

        if self.bad_eval_count >= self.patience:
            control.should_training_stop = True

        return control

moving_average_early_stopping = MovingAverageEarlyStoppingCallback(
    window_size=moving_average_window,
    patience=early_stopping_patience,
    threshold=early_stopping_threshold,
)

print("Early stopping will monitor a moving average of eval_reward.")
print(f"Moving-average window: {moving_average_window} evaluation rounds")
print(f"Patience: {early_stopping_patience} evaluation rounds without moving-average improvement")
print(f"Threshold: {early_stopping_threshold}")


Early stopping will monitor a moving average of eval_reward.
Moving-average window: 3 evaluation rounds
Patience: 2 evaluation rounds without moving-average improvement
Threshold: 0.0


In [17]:
training_args = GRPOConfig(
    output_dir="grpo-arithmetic-lora-early-stopping-demo",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    num_train_epochs=8,
    logging_steps=10,
    learning_rate=5e-5,
    log_completions=False, #True,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="smoothed_reward",
    greater_is_better=True,
    save_total_limit=2,
)

print("Training is allowed to run up to 8 epochs so moving-average early stopping has room to trigger before the full budget is used.")


Training is allowed to run up to 8 epochs so moving-average early stopping has room to trigger before the full budget is used.


## Trainer Construction

Create the GRPO trainer by connecting the base model, reward functions, training dataset, validation dataset, LoRA configuration, and the moving-average early stopping callback.


In [18]:
trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    reward_funcs=[format_reward, correctness_reward],
    callbacks=[moving_average_early_stopping],
    peft_config=lora_config,
)


## Training And Saving

Run GRPO training with validation-based early stopping enabled, then save the resulting adapter weights so the best checkpoint can be reused later.


## Plain Log Text (No ANSI Colors)

Disable colorized terminal output so training log text is easier to read in notebook outputs.

In [19]:
import os
from IPython.display import HTML, display
import trl.trainer.grpo_trainer as grpo_trainer_module

# Make TRL/Rich log tables render without color styling.
os.environ["NO_COLOR"] = "1"

# Force notebook output text to black for both ANSI and HTML-rendered tables.
display(HTML("""
<style>
.jp-OutputArea .ansi-yellow-fg,
.jp-OutputArea .ansi-green-fg,
.jp-OutputArea .ansi-blue-fg,
.jp-OutputArea .ansi-magenta-fg,
.jp-OutputArea .ansi-cyan-fg,
.jp-OutputArea .ansi-red-fg,
.jp-OutputArea .ansi-bright-black-fg,
.jp-OutputArea .ansi-bright-red-fg,
.jp-OutputArea .ansi-bright-green-fg,
.jp-OutputArea .ansi-bright-yellow-fg,
.jp-OutputArea .ansi-bright-blue-fg,
.jp-OutputArea .ansi-bright-magenta-fg,
.jp-OutputArea .ansi-bright-cyan-fg,
.jp-OutputArea .ansi-bright-white-fg,
.jp-OutputArea span[style*="color"],
.jp-OutputArea pre[style*="color"],
.jp-OutputArea-output table,
.jp-OutputArea-output table *,
.jp-OutputArea-output pre,
.jp-OutputArea-output code {
    color: #000000 !important;
    text-decoration-color: #000000 !important;
}
</style>
"""))


def _print_prompt_completions_sample_plain(
    prompts,
    completions,
    rewards,
    advantages,
    step,
    num_samples=None,
    extra=None,
):
    """Replacement for TRL rich logger that prints plain text with no color."""
    extra = extra or {}

    rows = []
    for i in range(len(prompts)):
        row = {
            "prompt": str(prompts[i]),
            "completion": str(completions[i]),
            "advantage": f"{advantages[i]:.2f}",
        }
        for reward_name, reward_values in rewards.items():
            row[reward_name] = f"{reward_values[i]:.2f}"
        for extra_name, extra_values in extra.items():
            row[extra_name] = str(extra_values[i])
        rows.append(row)

    if num_samples is not None and 0 < num_samples < len(rows):
        rows = rows[:num_samples]

    print(f"\n=== Step {step} completions (plain text) ===")
    for idx, row in enumerate(rows, start=1):
        print("-" * 80)
        print(f"Row {idx}")
        print(f"Prompt: {row['prompt']}")
        print(f"Completion: {row['completion']}")
        for key, value in row.items():
            if key not in {"prompt", "completion"}:
                print(f"{key}: {value}")


# Monkeypatch TRL GRPO logging to avoid Rich colorized tables.
# grpo_trainer_module.print_prompt_completions_sample = _print_prompt_completions_sample_plain

# print("Applied plain-text GRPO completion logger (no color). Re-run trainer.train() to use it.")

In [20]:
trainer.train()
trainer.save_model("grpo-arithmetic-lora-early-stopping-adapter")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/format Reward/mean,Rewards/format Reward/std,Rewards/correctness Reward/mean,Rewards/correctness Reward/std,Reward,Reward Std,Frac Reward Zero Std,Entropy,Clip Ratio/low Mean,Clip Ratio/low Min,Clip Ratio/high Mean,Clip Ratio/high Max,Clip Ratio/region Mean,Smoothed Reward,Smoothed Reward Window,Smoothed Reward Best,Smoothed Reward Bad Eval Count
10,0.000000,0.000000,27744.000000,64.000000,64.000000,64.000000,1.000000,0.000000,0.000000,0.000000,0.495833,0.011785,0.991667,0.023570,1.487500,0.035355,0.933333,1.209352,0.000000,0.000000,0.000000,0.000000,0.000000,1.487500,1,1.487500,0
20,-0.000000,-0.000000,55480.000000,64.000000,64.000000,64.000000,1.000000,0.000000,0.000000,0.000000,0.495833,0.011785,0.991667,0.023570,1.487500,0.035355,0.966667,0.842297,0.000000,0.000000,0.000000,0.000000,0.000000,1.487500,2,1.487500,1
30,0.000000,0.000000,83216.000000,64.000000,64.000000,64.000000,1.000000,0.000000,0.000000,0.000000,0.495833,0.011785,1.000000,0.000000,1.495833,0.011785,0.966667,0.952432,0.000000,0.000000,0.000000,0.000000,0.000000,1.490278,3,1.490278,0
40,0.000000,0.000000,110944.000000,64.000000,64.000000,64.000000,1.000000,0.000000,0.000000,0.000000,0.500000,0.000000,1.000000,0.000000,1.500000,0.000000,1.000000,1.034882,0.000000,0.000000,0.000000,0.000000,0.000000,1.494444,3,1.494444,0
50,0.000000,0.000000,138700.000000,64.000000,64.000000,64.000000,0.991667,4.266667,4.266667,4.266667,0.500000,0.000000,0.991667,0.023570,1.491667,0.023570,0.966667,0.810522,0.000000,0.000000,0.000000,0.000000,0.000000,1.495833,3,1.495833,0
60,0.000000,0.000000,166432.000000,64.000000,64.000000,64.000000,0.991667,4.266667,4.266667,4.266667,0.500000,0.000000,1.000000,0.000000,1.500000,0.000000,1.000000,0.922305,0.000000,0.000000,0.000000,0.000000,0.000000,1.497222,3,1.497222,0
70,0.000000,0.000000,194168.000000,64.000000,64.000000,64.000000,1.000000,0.000000,0.000000,0.000000,0.491667,0.023570,0.991667,0.023570,1.483333,0.047140,0.933333,0.495705,0.000000,0.000000,0.000000,0.000000,0.000000,1.491667,3,1.497222,1
80,0.000000,0.000000,221892.000000,64.000000,64.000000,64.000000,0.991667,4.266667,4.266667,4.266667,0.500000,0.000000,1.000000,0.000000,1.500000,0.000000,1.000000,0.420928,0.000000,0.000000,0.000000,0.000000,0.000000,1.494444,3,1.497222,2


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(

## GRPO Training Metrics

Summarize the GRPO-native training and validation metrics captured during the run so the moving-average early-stopping decision can be interpreted with reward, KL, entropy, clipping, raw validation reward, and smoothed validation reward trends.


In [21]:
grpo_logs = [row for row in trainer.state.log_history if "reward" in row and "eval_reward" not in row]
eval_logs = [row for row in trainer.state.log_history if "eval_reward" in row]

metric_columns = [
    ("reward", "reward"),
    ("reward_std", "reward_std"),
    ("rewards/format_reward/mean", "format_reward"),
    ("rewards/correctness_reward/mean", "correctness_reward"),
    ("kl", "kl"),
    ("entropy", "entropy"),
    ("clip_ratio/region_mean", "clip_ratio"),
]

eval_metric_columns = [
    ("eval_reward", "eval_reward"),
    ("eval_smoothed_reward", "eval_smoothed_reward"),
    ("eval_smoothed_reward_best", "best_smoothed_reward"),
    ("eval_smoothed_reward_bad_eval_count", "bad_eval_count"),
    ("eval_reward_std", "eval_reward_std"),
    ("eval_rewards/format_reward/mean", "eval_format_reward"),
    ("eval_rewards/correctness_reward/mean", "eval_correctness_reward"),
]

def format_value(value: float | None) -> str:
    if value is None:
        return "-"
    if isinstance(value, int):
        return str(value)
    return f"{value:.4f}"

if not grpo_logs:
    print("No GRPO training metric rows were found in trainer.state.log_history.")
else:
    available_columns = [
        (key, label)
        for key, label in metric_columns
        if any(key in row for row in grpo_logs)
    ]

    header = ["step"] + [label for _, label in available_columns]
    widths = {name: max(len(name), 12) for name in header}

    print("GRPO training metrics by logging step:")
    print("  " + "  ".join(name.ljust(widths[name]) for name in header))

    for row in grpo_logs:
        rendered = {"step": format_value(row.get("step"))}
        for key, label in available_columns:
            rendered[label] = format_value(row.get(key))

        print("  " + "  ".join(rendered[name].ljust(widths[name]) for name in header))

    final_row = grpo_logs[-1]
    print("\nFinal GRPO training snapshot:")
    for key, label in available_columns:
        print(f"  {label}: {format_value(final_row.get(key))}")

if not eval_logs:
    print("\nNo GRPO evaluation metric rows were found in trainer.state.log_history.")
else:
    available_eval_columns = [
        (key, label)
        for key, label in eval_metric_columns
        if any(key in row for row in eval_logs)
    ]

    eval_header = ["step"] + [label for _, label in available_eval_columns]
    eval_widths = {name: max(len(name), 18) for name in eval_header}

    print("\nGRPO validation metrics by evaluation step:")
    print("  " + "  ".join(name.ljust(eval_widths[name]) for name in eval_header))

    for row in eval_logs:
        rendered = {"step": format_value(row.get("step"))}
        for key, label in available_eval_columns:
            rendered[label] = format_value(row.get(key))

        print("  " + "  ".join(rendered[name].ljust(eval_widths[name]) for name in eval_header))

    best_eval_row = max(eval_logs, key=lambda row: row.get("eval_smoothed_reward", float("-inf")))
    print("\nBest validation snapshot by smoothed reward:")
    for key, label in available_eval_columns:
        print(f"  {label}: {format_value(best_eval_row.get(key))}")


GRPO training metrics by logging step:
  step          reward        reward_std    format_reward  correctness_reward  entropy       clip_ratio  
  10            1.1219        0.3769        0.3719         0.7500              1.1225        0.0000      
  20            1.4563        0.1205        0.4875         0.9688              1.0564        0.0000      
  30            1.4937        0.0354        0.4969         0.9969              0.8813        0.0000      
  40            1.4984        0.0088        0.4984         1.0000              0.9860        0.0000      
  50            1.4984        0.0088        0.4984         1.0000              0.9614        0.0000      
  60            1.4969        0.0177        0.5000         0.9969              0.7373        0.0000      
  70            1.4937        0.0354        0.5000         0.9938              0.7084        0.0000      
  80            1.4984        0.0088        0.4984         1.0000              0.3227        0.0000      

Final 

## Early Stopping Summary

Report which checkpoint won on smoothed validation reward and whether training stopped before the maximum planned budget.


In [22]:
best_checkpoint = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric
best_global_step = trainer.state.best_global_step
completed_step = trainer.state.global_step
max_steps = trainer.state.max_steps
stopped_early = completed_step < max_steps
smoothed_best_step = moving_average_early_stopping.best_step

print("Best validation checkpoint:", best_checkpoint or "None recorded")
print("Best smoothed validation reward:", "-" if best_metric is None else f"{best_metric:.4f}")
print("Best global step from Trainer state:", best_global_step if best_global_step is not None else "-")
print("Best smoothed-reward step from callback:", smoothed_best_step if smoothed_best_step is not None else "-")
print(f"Completed steps: {completed_step} / {max_steps}")

if stopped_early:
    print("Training stopped early because the moving average of validation reward stopped improving within the patience window.")
else:
    print("Training reached the configured maximum budget before moving-average early stopping was triggered.")


Best validation checkpoint: grpo-arithmetic-lora-early-stopping-demo/checkpoint-60
Best smoothed validation reward: 1.4972
Best global step from Trainer state: 60
Best smoothed-reward step from callback: 60
Completed steps: 80 / 120
Training stopped early because the moving average of validation reward stopped improving within the patience window.


## Filtered Training Completions (First 5 Train Records)

Load GRPO completion logs and display only rows tied to the first five training prompts.

In [23]:
completion_dir = Path(training_args.output_dir) / "completions"
completion_files = sorted(completion_dir.glob("completions_*.parquet"))

if not completion_files:
    print("No completion parquet files found. Ensure training finished with log_completions=True.")
else:
    completion_ds = Dataset.from_parquet([str(path) for path in completion_files])

    filtered_rows = [
        row for row in completion_ds
        if row.get("prompt") in first_five_train_prompts
    ]

    if not filtered_rows:
        print("No completion rows matched the first five training prompts.")
    else:
        print(f"Matched {len(filtered_rows)} rows for first 5 training prompts across all steps.")
        print("-" * 120)

        for i, row in enumerate(filtered_rows, start=1):
            prompt = row.get("prompt", "")
            completion = row.get("completion", "")
            step = row.get("step", "-")
            reward = row.get("reward", "-")
            format_reward_value = row.get("rewards/format_reward", row.get("rewards/format_reward/mean", "-"))
            correctness_reward_value = row.get("rewards/correctness_reward", row.get("rewards/correctness_reward/mean", "-"))

            print(f"Row {i} | step={step} | reward={reward} | format={format_reward_value} | correctness={correctness_reward_value}")
            print(f"Prompt: {prompt}")
            print(f"Completion: {completion}")
            print("-" * 120)

No completion parquet files found. Ensure training finished with log_completions=True.


## Post-Training Evaluation

Evaluate the restored best model on the held-out test dataset to compare behavior before and after GRPO and LoRA fine-tuning with early stopping.


In [24]:
# grab the trained model
trained_model = trainer.model

evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After GRPO + LoRA (Best Validation Checkpoint)"
)



=== After GRPO + LoRA (Best Validation Checkpoint) ===
Answer accuracy:   29/50 = 58.00%
Format compliance: 29/50 = 58.00%
Average reward:    0.870

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3 = 19</think>
<answer>19</answer>
Predicted: 19
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: Think: 3 + 7 = 10

Answer: 10
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6 = 18</think>
<answer>18</answer>
Predicted: 18
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <th